# configuration

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoConfig
from accelerate import Accelerator


In [ ]:
print(os.getcwd())
if os.getcwd() != "/gpfs/share/home/2300012116/lifespanResearch/model":
    os.chdir("/gpfs/share/home/2300012116/lifespanResearch/model")
print(os.getcwd())

In [ ]:
# --- 导入自定义模块 ---
from module.dataset import ProteinDataset
from module.train import train_one_epoch_mask_predict, evaluate_mask_predict
from module.model import ESMCForMaskedLM
from module.collator import ESMCDataCollator
from module.utils import load_local_esmc

In [ ]:
import datetime
from torch.utils.tensorboard import SummaryWriter

In [ ]:

# ================= 配置参数 =================
CSV_PATH = "fpbase_data/original_data.csv"          # 数据文件路径
MODEL_NAME = "EvolutionaryScale/esmc-600m" # 替换为你具体使用的模型名称/路径
SAVE_PATH = "/gpfs/share/home/2300012116/lifespanResearch/esmc-600m-2024-12" # 模型保存路径
UPDATE_PATH = "/gpfs/share/home/2300012116/lifespanResearch/evolveModel/embed_weight"
LOG_PATH = "/gpfs/share/home/2300012116/lifespanResearch/evolveModel/log"

BATCH_SIZE = 16                 # 显存够大可以调大 (e.g., 8, 16)
EPOCHS = 20                    # 训练轮数
LEARNING_RATE = 5e-5           # 学习率
WEIGHT_DECAY = 0.05           # 权重衰减
MAX_LENGTH = 1024              # 序列最大长度
MASK_RATIO = 0.15              # 掩码比例

# ===========================================

In [ ]:
accelerator = Accelerator()
device = accelerator.device
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

In [ ]:
esmc_model,tokenizer= load_local_esmc(device=device, root=SAVE_PATH)


In [ ]:
from esm.models.esmc import ESMC
from esm.sdk.api import (
    ESM3InferenceClient,
    ESMProtein,
    ESMProteinError,
    LogitsConfig,
    LogitsOutput,
    ProteinType,
)
from esm.tokenization import get_esm3_model_tokenizers, get_esmc_model_tokenizers
from esm.utils.constants.esm3 import data_root
from typing import List, Tuple
from esm.utils.sampling import _BatchedESMProteinTensor

In [ ]:
protein = ESMProtein(sequence="AAAAA")
protein_tensor = esmc_model.encode(protein)
logits_output = esmc_model.logits(
   protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
)
print(logits_output.logits, logits_output.embeddings)

# construct the dataset

In [ ]:
# 3. 准备数据
# 实例化 Dataset (读取 CSV)
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"找不到数据文件: {CSV_PATH}，请确保文件在正确位置。")

full_dataset = ProteinDataset(CSV_PATH, max_length=MAX_LENGTH)


print(f"数据集总量: {len(full_dataset)} 条序列")


In [ ]:
# 划分训练集 (90%) 和验证集 (10%)
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])


In [ ]:
   # 4. 初始化 Data Collator (数据处理器)
    # 这就是我们之前写的那个类，它负责 Tokenization 和 Masking
collator = ESMCDataCollator(tokenizer, mask_ratio=0.15)



In [ ]:
    # 5. 创建 DataLoader
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,          # 训练集必须打乱
    collate_fn=collator    # 关键：挂载处理器
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,         # 验证集不打乱
    collate_fn=collator
)

In [ ]:
# 6. 初始化模型封装 (Model Wrapper)
    # 这里的 model 才是我们要训练的对象
model = ESMCForMaskedLM(esmc_model)
model.to(device)

# 7. 设置优化器
    # 过滤掉不需要梯度的参数 (通常不需要，除非你冻结了层)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

In [ ]:
# 8. 读取以前的模型权重（如果有）
save_path = os.path.join(SAVE_PATH, "embed_weight.pt")
if os.path.exists(save_path):
    print(f"发现先前的模型权重，正在加载: {save_path}")
    model.load_state_dict(torch.load(save_path, map_location=device)) #is a directory
else:
    print("没有找到之前的模型权重，将从头开始训练。")

In [ ]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
# 假设 UPDATE_PATH 是你保存模型权重的目录
# 我们在 UPDATE_PATH 下创建一个 logs 子目录，并带上时间戳，防止覆盖之前的实验
log_dir = os.path.join(LOG_PATH, "logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
writer = SummaryWriter(log_dir=log_dir)  # 2. 初始化 Writer

# 9. 开始训练循环
best_val_loss = float('inf')


for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    train_loss = train_one_epoch_mask_predict(model, train_loader, optimizer, accelerator, epoch+1)
    print(f"训练集 Loss: {train_loss:.4f}")
    
    writer.add_scalar('Loss/Train', train_loss, epoch + 1)
    

    if len(val_dataset) > 0:
        val_loss, val_acc = evaluate_mask_predict(model, val_loader, accelerator)
        print(f"验证集 Loss: {val_loss:.4f} | 掩码准确率 (Masked Acc): {val_acc:.2f}%")
        
        writer.add_scalar('Loss/Validation', val_loss, epoch + 1)
        writer.add_scalar('Accuracy/Masked_Acc', val_acc, epoch + 1)
        
        writer.add_scalars('Loss/Combined', {
            'train': train_loss,
            'val': val_loss
        }, epoch + 1)

        # 保存最佳模型
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_path = os.path.join(UPDATE_PATH, "embed_weight.pt")
            torch.save(model.state_dict(), save_path)
            print(f"最佳模型，已保存至: {save_path}")
    else:
        print("验证集为空，跳过验证。")
        
    # --- 新增: 记录当前学习率 (如果用了 Scheduler 很有用) ---
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Learning_Rate', current_lr, epoch + 1)
    
    # 强制将数据写入磁盘
    writer.flush()

# 训练结束关闭 writer
writer.close()
print("\n训练结束！")


Train Epoch 1:  23%|████████████▋                                          | 12/52 [00:15<00:53,  1.33s/it, loss=1.9340]

Train Epoch 1:  23%|████████████▋                                          | 12/52 [00:15<00:53,  1.33s/it, loss=1.9340]


KeyboardInterrupt: 